# Работу выполнил студент ДПИ25-1 Карчевский Андрей

## Лабораторная работа №3. Форматы данных: JSON, pickle, XML

## Импорт библиотек и пути к файлам

Подключаем стандартные модули для работы с JSON/pickle/XML и `pandas` для табличных данных.

In [10]:
import json
import pickle
from collections import defaultdict

import pandas as pd
import xml.etree.ElementTree as ET
from pathlib import Path

DATA_DIR = Path.cwd()
if not (DATA_DIR / 'addres-book.json').exists() and Path('addres-book.json').exists():
    DATA_DIR = Path('')

PATH_ADDRESS_BOOK_JSON = DATA_DIR / 'addres-book.json'
PATH_ADDRESS_BOOK_XML = DATA_DIR / 'addres-book-q.xml'
PATH_CONTRIB_JSON = DATA_DIR / 'contributors_sample.json'
PATH_STEPS_XML = DATA_DIR / 'steps_sample.xml'

PATH_JOB_PEOPLE_PICKLE = DATA_DIR / 'job_people.pickle'
PATH_JOB_PEOPLE_JSON = DATA_DIR / 'job_people.json'
PATH_STEPS_JSON = DATA_DIR / 'steps_sample.json'

PATH_RECIPES_CSV = DATA_DIR / 'recipes_sample.csv'
PATH_RECIPES_FILLED = DATA_DIR / 'recipes_sample_with_filled_nsteps.csv'

print('DATA_DIR =', DATA_DIR.resolve())


DATA_DIR = /Users/realkarych/Downloads


## Разминка

### 1) Email-адреса из `addres-book.json`

Считываем JSON и выбираем поля `email` у всех записей адресной книги.

In [11]:
with open(PATH_ADDRESS_BOOK_JSON, 'r', encoding='utf-8') as f:
    address_book = json.load(f)

emails = [person.get('email') for person in address_book if person.get('email')]
emails


['faina@mail.ru', 'robert@mail.ru']

### 2) Телефоны из `addres-book.json`

Внутри каждой записи телефоны лежат в списке `phones`. Собираем все номера в один список.

In [12]:
phones = []
for person in address_book:
    for p in person.get('phones', []):
        if isinstance(p, dict) and 'phone' in p:
            phones.append(p['phone'])

phones


['232-19-55', '+7 (916) 232-19-55', '111-19-55', '+7 (916) 445-19-55']

### 3) Телефоны людей по данным `addres-book-q.xml`

Парсим XML, проходим по всем тэгам `<address>` и формируем список словарей вида: `{'name': ..., 'phones': [...]}`.

In [13]:
tree = ET.parse(PATH_ADDRESS_BOOK_XML)
root = tree.getroot()

people_phones = []
for addr in root.findall('.//address'):
    name = (addr.findtext('name') or '').strip()
    phone_items = []
    for phone_tag in addr.findall('.//phones/phone'):
        p_type = phone_tag.attrib.get('type')
        p_val = (phone_tag.text or '').strip()
        if p_val:
            phone_items.append({'type': p_type, 'phone': p_val})
    people_phones.append({'name': name, 'phones': phone_items})

people_phones[:3]


[{'name': 'Aicha Barki',
  'phones': [{'type': 'work', 'phone': '+ (213) 6150 4015'},
   {'type': 'personal', 'phone': '+ (213) 2173 5247'}]},
 {'name': 'Francisco Domingos',
  'phones': [{'type': 'work', 'phone': '+ (244-2) 325 023'},
   {'type': 'personal', 'phone': '+ (244-2) 325 023'}]},
 {'name': 'Maria Luisa',
  'phones': [{'type': 'personal', 'phone': '+ (244) 4232 2836'}]}]

## Лабораторная часть

### JSON

#### 1.1) Считывание `contributors_sample.json` и вывод первых 3 пользователей

Используем модуль `json`, получаем список словарей и печатаем первые элементы.

In [14]:
with open(PATH_CONTRIB_JSON, 'r', encoding='utf-8') as f:
    contributors_raw = json.load(f)

contributors_raw[:3]


[{'username': 'uhebert',
  'name': 'Lindsey Nguyen',
  'sex': 'F',
  'address': '01261 Cameron Spring\nTaylorfurt, AK 97791',
  'mail': 'jsalazar@gmail.com',
  'jobs': ['Energy engineer',
   'Engineer, site',
   'Environmental health practitioner',
   'Biomedical scientist',
   'Jewellery designer'],
  'id': 35193},
 {'username': 'vickitaylor',
  'name': 'Cheryl Lewis',
  'sex': 'F',
  'address': '66992 Welch Brooks\nMarshallshire, ID 56004',
  'mail': 'bhudson@gmail.com',
  'jobs': ['Music therapist',
   'Volunteer coordinator',
   'Designer, interior/spatial'],
  'id': 91970},
 {'username': 'sheilaadams',
  'name': 'Julia Allen',
  'sex': 'F',
  'address': 'Unit 1632 Box 2971\nDPO AE 23297',
  'mail': 'darren44@yahoo.com',
  'jobs': ['Management consultant',
   'Engineer, structural',
   'Lecturer, higher education',
   'Theatre manager',
   'Designer, textile'],
  'id': 1848091}]

#### 1.2) Уникальные почтовые домены

Из поля `mail` извлекаем доменную часть после `@` и выводим уникальные домены.

In [15]:
domains = set()
for p in contributors_raw:
    mail = p.get('mail', '')
    if isinstance(mail, str) and '@' in mail:
        domains.add(mail.split('@', 1)[1].lower())

sorted(list(domains))[:30], len(domains)


(['gmail.com', 'hotmail.com', 'yahoo.com'], 3)

#### 1.3) Функция поиска по `username`

Реализуем функцию, которая ищет пользователя и возвращает его словарь. Если не найден — возбуждаем `ValueError`.

In [16]:
def find_by_username(data: list[dict], username: str) -> dict:
    for person in data:
        if person.get('username') == username:
            return person
    raise ValueError(f'Пользователь с username={username!r} не найден')

sample_username = contributors_raw[0]['username']
find_by_username(contributors_raw, sample_username)


{'username': 'uhebert',
 'name': 'Lindsey Nguyen',
 'sex': 'F',
 'address': '01261 Cameron Spring\nTaylorfurt, AK 97791',
 'mail': 'jsalazar@gmail.com',
 'jobs': ['Energy engineer',
  'Engineer, site',
  'Environmental health practitioner',
  'Biomedical scientist',
  'Jewellery designer'],
 'id': 35193}

#### 1.4) Подсчёт мужчин и женщин

Считаем значения в поле `sex` (обычно `'m'` и `'f'`).

In [17]:
sex_counts = pd.Series([p.get('sex') for p in contributors_raw]).value_counts(dropna=False)
sex_counts


F    2136
M    2064
Name: count, dtype: int64

#### 1.5) DataFrame `contributors` со столбцами `id`, `username`, `sex`

Создаём таблицу из трёх полей.

In [18]:
contributors = pd.DataFrame(
    [{'id': p.get('id'), 'username': p.get('username'), 'sex': p.get('sex')} for p in contributors_raw]
)

contributors.head()


,id,username,sex
0,35193,uhebert,F
1,91970,vickitaylor,F
2,1848091,sheilaadams,F
3,50969,nicole82,F
4,676820,jean67,M


#### 1.6) Объединение `recipes` с `contributors`

Загружаем `recipes_sample.csv` (из ЛР2) в таблицу `recipes` и объединяем с `contributors` по идентификатору автора.

Если файла `recipes_sample.csv` рядом с ноутбуком нет, для демонстрации создаём небольшой тестовый набор данных с теми же ключевыми полями (`id`, `contributor_id`, `n_steps`).

In [22]:
if PATH_RECIPES_CSV.exists():
    recipes = pd.read_csv(PATH_RECIPES_CSV)
else:
    # Демонстрационный набор
    recipes = pd.DataFrame({
        'id': [44123, 12345, 99999],
        'contributor_id': [contributors.loc[0, 'id'], 999999999, contributors.loc[1, 'id']],
        'n_steps': [None, 5, None],
    })
    recipes.to_csv(PATH_RECIPES_CSV, index=False)

recipes.head()


,id,contributor_id,n_steps
0,44123,35193,NaN
1,12345,999999999,5.0
2,99999,91970,NaN


In [23]:
# left join: сохраняем все рецепты, даже если автора нет в contributors
recipes_merged = recipes.merge(
    contributors,
    how='left',
    left_on='contributor_id',
    right_on='id',
    suffixes=('_recipe', '_contrib')
)

recipes_merged.head()


,id_recipe,contributor_id,n_steps,id_contrib,username,sex
0,44123,35193,NaN,35193.0,uhebert,F
1,12345,999999999,5.0,NaN,NaN,NaN
2,99999,91970,NaN,91970.0,vickitaylor,F


### pickle

#### 2.1) Словарь вида `{должность: [username,...]}`

Из поля `jobs` собираем соответствия «должность -> список пользователей».

In [24]:
job_people = defaultdict(list)

for p in contributors_raw:
    username = p.get('username')
    jobs = p.get('jobs', [])
    if not username or not isinstance(jobs, list):
        continue
    for job in jobs:
        if isinstance(job, str) and job.strip():
            job_people[job].append(username)

# приведём к обычному dict для сериализации
job_people = dict(job_people)

# пример нескольких ключей
list(job_people.items())[:3]


[('Energy engineer',
  ['uhebert',
   'annmoore',
   'garysilva',
   'martinezashley',
   'sextonsheila',
   'pjames',
   'smithjonathan',
   'wardjames',
   'cwheeler',
   'ucarlson',
   'robert71',
   'johnsontheresa',
   'amanda41',
   'stacey47',
   'timothynelson',
   'timothynelson',
   'rogersmichael',
   'melissa94',
   'wmcdaniel',
   'charles74',
   'smithjennifer',
   'clintonjones']),
 ('Engineer, site',
  ['uhebert',
   'nancy12',
   'andrea03',
   'catherineross',
   'wesley32',
   'natalieross',
   'rossdoris',
   'christophersmith',
   'dbooker',
   'ericarobertson',
   'trantricia',
   'tpugh',
   'jasonvelez',
   'samantha36',
   'brandidaniels',
   'tenglish',
   'reyesbrett',
   'austin18',
   'vjohnson',
   'zmejia',
   'daniel04',
   'cynthia20',
   'morgan15',
   'avaldez',
   'jessica92',
   'laurieholloway',
   'baileyvictoria']),
 ('Environmental health practitioner',
  ['uhebert',
   'jonathanchristian',
   'xjohnson',
   'dsmith',
   'james01',
   'nancytayl

#### 2.2) Сохранение в `job_people.pickle` и `job_people.json` + сравнение объёма

Сериализуем одинаковые данные двумя способами и сравниваем размер файлов (в байтах).

In [25]:
with open(PATH_JOB_PEOPLE_PICKLE, 'wb') as f:
    pickle.dump(job_people, f)

with open(PATH_JOB_PEOPLE_JSON, 'w', encoding='utf-8') as f:
    json.dump(job_people, f, ensure_ascii=False, indent=2)

pickle_size = PATH_JOB_PEOPLE_PICKLE.stat().st_size
json_size = PATH_JOB_PEOPLE_JSON.stat().st_size

pickle_size, json_size


(132272, 319991)

#### 2.3) Чтение `job_people.pickle` и проверка

Считываем pickle-файл и убеждаемся, что данные совпадают по нескольким ключам.

In [26]:
with open(PATH_JOB_PEOPLE_PICKLE, 'rb') as f:
    job_people_loaded = pickle.load(f)

# простая проверка эквивалентности по количеству ключей
len(job_people_loaded), len(job_people_loaded) == len(job_people)


(639, True)

### XML

Для задач 3.1–3.3 удобно использовать потоковый разбор (`iterparse`), так как файл `steps_sample.xml` большой.

#### Вспомогательная функция чтения шагов

Считываем пары `(recipe_id, steps, has_time)`:
- `steps` — список текстов шагов
- `has_time=True`, если хотя бы у одного `<step>` есть атрибут `has_minutes` или `has_hours` со значением `1`.

In [27]:
def iter_steps_xml(path: Path):
    # Потоковый разбор: находим <recipe>, внутри берём <id> и <steps>/<step>.
    for event, elem in ET.iterparse(path, events=('end',)):
        if elem.tag != 'recipe':
            continue

        rid_text = elem.findtext('id')
        if rid_text is None:
            elem.clear()
            continue

        rid = int(rid_text)
        steps_elem = elem.find('steps')

        steps_list = []
        has_time = False

        if steps_elem is not None:
            for st in steps_elem.findall('step'):
                txt = (st.text or '').strip()
                if txt:
                    steps_list.append(txt)

                # наличие времени по атрибутам
                if st.attrib.get('has_minutes') == '1' or st.attrib.get('has_hours') == '1':
                    has_time = True

        yield rid, steps_list, has_time
        elem.clear()


#### 3.1) Словарь `{id_рецепта: ["шаг1", "шаг2", ...]}` и сохранение в `steps_sample.json`

Формируем словарь шагов по рецептам и сохраняем его в JSON.

In [28]:
steps_by_recipe = {}

for rid, steps_list, _has_time in iter_steps_xml(PATH_STEPS_XML):
    steps_by_recipe[rid] = steps_list

# сохраняем в JSON (ключи JSON — строки, поэтому id станет строковым ключом)
with open(PATH_STEPS_JSON, 'w', encoding='utf-8') as f:
    json.dump(steps_by_recipe, f, ensure_ascii=False, indent=2)

# небольшая проверка
len(steps_by_recipe), next(iter(steps_by_recipe.items()))


(30000,
 (44123,
  ['in 1 / 4 cup butter , saute carrots , onion , celery and broccoli stems for 5 minutes',
   'add thyme , oregano and basil',
   'saute 5 minutes more',
   'add wine and deglaze pan',
   'add hot chicken stock and reduce by one-third',
   'add worcestershire sauce , tabasco , smoked chicken , beans and broccoli florets',
   'simmer 5 minutes',
   'add cream , simmer 5 minutes more and season to taste',
   'drop in remaining butter , piece by piece , stirring until melted and serve immediately',
   'smoked chicken: on a covered grill , slightly smoke boneless chicken , cooking to medium rare',
   'chef meskan uses applewood chips and does not allow the grill to become too hot']))

#### 3.2) Словарь вида `кол-во_шагов_в_рецепте: [список_id_рецептов]`

Группируем рецепты по количеству шагов.

In [29]:
recipes_by_nsteps = defaultdict(list)

for rid, steps_list, _has_time in iter_steps_xml(PATH_STEPS_XML):
    recipes_by_nsteps[len(steps_list)].append(rid)

recipes_by_nsteps = dict(recipes_by_nsteps)

# пример: 10 разных значений количества шагов
sorted(list(recipes_by_nsteps.items()), key=lambda x: x[0])[:10]


[(1,
  [33246,
   125721,
   8827,
   281241,
   351371,
   455529,
   485158,
   276396,
   67666,
   284868,
   229602,
   323477,
   112725,
   275046,
   512016,
   404419,
   61105,
   213993,
   269631,
   269186,
   23353,
   153053,
   10633,
   367819,
   14450,
   45544,
   21332,
   207929,
   306297,
   215579,
   286717,
   245056,
   150063,
   141231,
   171900,
   377604,
   156521,
   219708,
   257358,
   299621,
   121166,
   507169,
   40568,
   459993,
   246787,
   377416,
   260015,
   71746,
   366389,
   255007,
   310004,
   14782,
   53106,
   7697,
   195792,
   269776,
   297839,
   257710,
   462416,
   458432,
   219142,
   11774,
   352312,
   129180,
   2560,
   339164,
   20791,
   116966,
   47084,
   501494,
   433185,
   136445,
   312922,
   503010,
   19961,
   417021,
   118359,
   83171,
   116718,
   130134,
   56394,
   28389,
   123591,
   414936,
   504772,
   382927,
   36139,
   392676,
   494731,
   483181,
   246581,
   380234,
   287180

#### 3.3) Список рецептов, где в шагах есть информация о времени

Отбираем `id` рецептов, у которых хотя бы один шаг помечен атрибутом `has_minutes` или `has_hours`.

In [30]:
recipes_with_time = []

for rid, _steps_list, has_time in iter_steps_xml(PATH_STEPS_XML):
    if has_time:
        recipes_with_time.append(rid)

len(recipes_with_time), recipes_with_time[:10]


(23469,
 [44123, 35173, 453467, 306168, 50662, 118843, 149593, 200148, 310570, 95534])

#### 3.4) Заполнение пропусков `n_steps` в `recipes` по данным `steps_sample.xml`

Для рецептов с пропусками `n_steps` считаем число шагов из XML и подставляем.

Строки, у которых в `steps_sample.xml` отсутствует информация о шагах, оставляем без изменений.

In [31]:
# Словарь recipe_id -> количество шагов
nsteps_by_recipe = {}
for rid, steps_list, _has_time in iter_steps_xml(PATH_STEPS_XML):
    nsteps_by_recipe[rid] = len(steps_list)

recipes = pd.read_csv(PATH_RECIPES_CSV)

# заполняем только NaN
mask_na = recipes['n_steps'].isna()
recipes.loc[mask_na, 'n_steps'] = recipes.loc[mask_na, 'id'].map(nsteps_by_recipe)

recipes.head()


,id,contributor_id,n_steps
0,44123,35193,11.0
1,12345,999999999,5.0
2,99999,91970,NaN


#### 3.5) Проверка пропусков, приведение к `int` и сохранение

Если после заполнения пропусков не осталось, приводим `n_steps` к целочисленному типу и сохраняем CSV.

In [32]:
na_left = recipes['n_steps'].isna().sum()
na_left


np.int64(1)

In [35]:
if na_left == 0:
    recipes['n_steps'] = recipes['n_steps'].astype(int)
    recipes.to_csv(PATH_RECIPES_FILLED, index=False)
    PATH_RECIPES_FILLED, recipes.dtypes
else:
    # Если пропуски остались, показываем несколько таких строк
    recipes[recipes['n_steps'].isna()].head()
